# DARL PhysioNet A → B — v1

A: XGBoost con pacientes disjuntos 70/15/15. B: UCI simulada de 200 camas, predicción cada hora y decisión al cierre de cada día. ICULOS es relativo al paciente, no una fecha. Las etiquetas de B llegan con 24 horas de retraso.

## 1. Importaciones y semilla
Se usa la semilla 42 en particiones, colas y DQN.

In [ ]:
import json, sys, zipfile, shutil
from pathlib import Path
import numpy as np
import torch
np.random.seed(42)
torch.manual_seed(42)

## 2. Resolver los dos datasets
El paquete puede venir como directorio o como `darl.zip`.

In [ ]:
mounted = Path('/kaggle/input')
data_candidates = list(mounted.rglob('physionet.parquet'))
print('Montajes:', [p.name for p in mounted.iterdir()])
assert len(data_candidates) == 1, data_candidates
data_path = data_candidates[0]
package_candidates = list(mounted.rglob('darl.zip'))
directory_candidates = [p for p in mounted.rglob('bed_stream.py') if p.parent.name == 'data']
assert package_candidates or directory_candidates, 'darl package not mounted'
package_root = package_candidates[0].parent if package_candidates else directory_candidates[0].parent.parent
if (package_root / 'darl').is_dir():
    sys.path.insert(0, str(package_root))
elif (package_root / 'data' / 'bed_stream.py').is_file():
    shutil.copytree(package_root, Path('/kaggle/temp/darl'), dirs_exist_ok=True)
    sys.path.insert(0, '/kaggle/temp')
else:
    archive = package_root / 'darl.zip'
    assert archive.is_file(), list(package_root.iterdir())
    destination = Path('/kaggle/temp/darl')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    sys.path.insert(0, '/kaggle/temp')
from darl.evaluation.bed_experiment import run_experiment
import hashlib, darl
expected_hashes = {'data/bed_stream.py': 'c020e87407a20df6090c75860c901905a1a73b682382659a116c09ad3eb2e71b', 'evaluation/bed_experiment.py': '10bb666b9504c8bdeb2c56a180762c811f3add6c92c41632004fd1588d3f1221', 'rl/bed_env.py': '4e5e9f4df1dfc81e152c20818020f4016c10f9aebe664044da39531c430bf0b6', 'visualization/bed_figures.py': '5e68242a3f328aef99d968cb75090697c823ca4579cbbe43a40da9d357e243c2'}
package_dir = Path(next(iter(darl.__path__)))
for relative, expected in expected_hashes.items():
    actual = hashlib.sha256((package_dir / relative).read_bytes()).hexdigest()
    assert actual == expected, f'Paquete Kaggle desactualizado: {relative}'
assert torch.cuda.is_available(), 'T4/CUDA no disponible'
print('CUDA:', torch.cuda.get_device_name(0), '| parquet:', data_path.name)

## 3. DQN del laboratorio
Red Q de dos capas ocultas de 128 ReLU, replay FIFO, ε-greedy, Adam y copia rígida de la red objetivo cada 200 actualizaciones. Estado DARL de 28 componentes y cuatro acciones. Una transición entra al replay solo cuando se conoce la etiqueta que determina su recompensa.

In [ ]:
from darl.rl.course_dqn import CourseDQNConfig
cfg = CourseDQNConfig()
print({'state_dim': cfg.state_dim, 'actions': cfg.action_dim, 'hidden': cfg.hidden, 'target_interval': cfg.target_interval, 'device': 'cuda'})

## 4. Experimento
La versión corta verifica integración; la completa evalúa cuatro escenarios y cinco secuencias finales.

In [ ]:
output_dir = Path('/kaggle/working')
summary = run_experiment(data_path, output_dir, version='v1', device='cuda', seed=42)
print({key: summary[key] for key in ('status', 'version', 'n_test_runs', 'dqn_updates')})

## 5. Verificación de artefactos
Las métricas finales usan exclusivamente pacientes B reservados para prueba.

In [ ]:
assert summary['status'] == 'complete'
assert (output_dir / 'run_summary.json').is_file()
assert (output_dir / 'final_metrics.csv').is_file()
print('Archivos:', sorted(path.name for path in output_dir.iterdir() if path.is_file()))

## 6. Figuras para presentación
Ocupación, calendario del drift, AUPRC, recompensa y acciones del DQN. Se exportan PNG y SVG.

In [ ]:
from darl.visualization.bed_figures import make_presentation_figures
figures = make_presentation_figures(output_dir, seed=42)
summary['presentation_figures'] = [str(Path(item).relative_to(output_dir)) for item in figures]
(output_dir / 'run_summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
print('Figuras:', [Path(item).name for item in figures])